# Notebook 01: Extracción y Limpieza del Dataset TAWOS

## Objetivo
Conectar con la base de datos MySQL local (TAWOS) alojada en un contenedor Docker, extraer las variables de interés
mediante una consulta SQL desnormalizada, limpiar los datos y generar las **variables objetivo** (`Target_Retraso` y `Target_Riesgo`)
siguiendo las Reglas de Negocio (RN-01 y RN-02) del proyecto. El resultado es un dataset tabular limpio listo para el entrenamiento de modelos ML.

In [7]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, inspect

# Configuración de la conexión a MySQL (Docker local)
db_user = 'root'
db_pass = 'root'
db_host = 'localhost'
db_port = '3306'
db_name = 'tawos'

# Crear el motor de conexión
engine = create_engine(f'mysql+pymysql://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}')
print("Motor de base de datos inicializado.")

Motor de base de datos inicializado.


## 1. Exploración del esquema relacional
Inspeccionamos la base de datos TAWOS para entender su estructura (13 tablas, jerarquía Project → Sprint → Issue)
y planificar la extracción selectiva de variables.

In [8]:
# Usamos el 'engine' que ya creaste en la Celda 1
inspector = inspect(engine)

# Obtener todos los nombres de las tablas
tablas = inspector.get_table_names()

print(f"🔍 Exploración del archivo RAW (Base de datos 'tawos')")
print(f"Se han encontrado {len(tablas)} tablas.\n")
print("=" * 60)

for tabla in tablas:
    # Obtener las columnas de la tabla actual
    columnas = inspector.get_columns(tabla)
    nombres_columnas = [col['name'] for col in columnas]
    
    # Hacer una consulta rápida para ver cuántas filas tiene
    try:
        conteo = pd.read_sql(f"SELECT COUNT(*) FROM `{tabla}`", engine).iloc[0, 0]
    except Exception as e:
        conteo = "Error al leer"
        
    print(f"📌 TABLA: {tabla} | Filas: {conteo}")
    print(f"   Columnas ({len(nombres_columnas)}): {', '.join(nombres_columnas)}")
    print("-" * 60)

🔍 Exploración del archivo RAW (Base de datos 'tawos')
Se han encontrado 13 tablas.

📌 TABLA: Affected_Version | Filas: 264004
   Columnas (2): Issue_ID, Affected_Version_ID
------------------------------------------------------------
📌 TABLA: Change_Log | Filas: 9253419
   Columnas (10): ID, Field, From_Value, To_Value, From_String, To_String, Change_Type, Creation_Date, Author_ID, Issue_ID
------------------------------------------------------------
📌 TABLA: Comment | Filas: 1518327
   Columnas (7): ID, Comment, Comment_Text, Comment_Code, Creation_Date, Author_ID, Issue_ID
------------------------------------------------------------
📌 TABLA: Component | Filas: 2001
   Columnas (5): ID, Jira_ID, Name, Description, Project_ID
------------------------------------------------------------
📌 TABLA: Fix_Version | Filas: 224254
   Columnas (2): Issue_ID, Fix_Version_ID
------------------------------------------------------------
📌 TABLA: Issue | Filas: 458232
   Columnas (30): ID, Jira_ID, I

## 2. Extracción SQL desnormalizada
Ejecutamos una query que cruza las tablas principales (`Issue`, `Project`, `Sprint`, `Issue_Link`)
para obtener una vista plana con las 14 variables de interés. Se filtran tareas sin tiempo de resolución ni esfuerzo.

In [9]:
# 1. EXTRACCIÓN SQL
query = """
SELECT 
    i.Issue_Key,
    i.Type as Issue_Type,
    p.ID as Project_ID,
    p.Name as Project_Name,
    s.ID as Sprint_ID,
    s.State as Sprint_State,
    i.Story_Point,
    i.Total_Effort_Minutes,
    i.In_Progress_Minutes,
    i.Resolution_Time_Minutes,
    i.Title_Changed_After_Estimation,
    i.Description_Changed_After_Estimation,
    i.Story_Point_Changed_After_Estimation,
    COALESCE(links.Blocker_Count, 0) as Blocker_Count
FROM Issue i
LEFT JOIN Project p ON i.Project_ID = p.ID
LEFT JOIN Sprint s ON i.Sprint_ID = s.ID 
LEFT JOIN (
    SELECT 
        Issue_ID, 
        COUNT(*) as Blocker_Count
    FROM Issue_Link 
    WHERE Name LIKE '%%Block%%' OR Name LIKE '%%Depend%%' OR Description LIKE '%%Block%%'
    GROUP BY Issue_ID
) links ON i.ID = links.Issue_ID
WHERE i.Resolution_Time_Minutes IS NOT NULL 
  AND i.Total_Effort_Minutes IS NOT NULL;
"""

print("Ejecutando extracción desde MySQL...")
df_master = pd.read_sql(query, engine)
print(f"Extracción completada: {df_master.shape[0]} registros y {df_master.shape[1]} variables.")

Ejecutando extracción desde MySQL...
Extracción completada: 458232 registros y 14 variables.


## 3. Limpieza profunda y refinamiento
Se aplican tres fases secuenciales:
- **Imputación de nulos:** Banderas lógicas a 0, Sprints fantasma a `-1`/`UNASSIGNED`.
- **Filtrado de filas:** Se eliminan tareas con esfuerzo 0, resolución instantánea y outliers extremos (percentiles 98/99).
- **Transformación:** Agrupación semántica de tipos de tarea (ej. `Sub-task` → `Task`, `Improvement` → `Enhancement`).

In [10]:
# 2. LIMPIEZA PROFUNDA Y REFINAMIENTO

# ==========================================================
# FASE 1: IMPUTACIÓN DE NULOS (Missingness as a Feature)
# ==========================================================

# 1.1 Banderas lógicas y Magnitudes
columnas_flags = ['Title_Changed_After_Estimation', 'Description_Changed_After_Estimation', 'Story_Point_Changed_After_Estimation']
df_master[columnas_flags] = df_master[columnas_flags].fillna(0).astype(int)
df_master['Story_Point'] = df_master['Story_Point'].fillna(0)

# 1.2 Variables de Gestión (Sprints fantasma)
df_master['Sprint_ID'] = df_master['Sprint_ID'].fillna(-1).astype(int)
df_master['Sprint_State'] = df_master['Sprint_State'].fillna('UNASSIGNED')


# ==========================================================
# FASE 2: FILTRADO DE FILAS (Creación del dataset limpio)
# ==========================================================

# 2.1 Filtrar trampas (esfuerzo 0 o tareas instantáneas)
df_clean = df_master[
    (df_master['Total_Effort_Minutes'] > 0) & 
    (df_master['Resolution_Time_Minutes'] > 0)
].copy()

# 2.2 Filtrar Outliers extremos (quitamos tiempos irreales)
q_time = df_clean['Resolution_Time_Minutes'].quantile(0.98)
q_points = df_clean['Story_Point'].quantile(0.99)
df_clean = df_clean[
    (df_clean['Resolution_Time_Minutes'] <= q_time) &
    (df_clean['Story_Point'] <= q_points)
]


# ==========================================================
# FASE 3: TRANSFORMACIÓN DE VARIABLES
# ==========================================================

# 3.1 Agrupación y simplificación de Tipos de Tarea
mapeo_tipos = {
    'Bug': 'Bug', 'Story': 'Story', 'Epic': 'Epic',
    'Task': 'Task', 'Sub-task': 'Task', 'Technical task': 'Task',
    'New Feature': 'Enhancement', 'Improvement': 'Enhancement',
    'Suggestion': 'Enhancement', 'Enhancement Request': 'Enhancement'
}
df_clean['Issue_Type'] = df_clean['Issue_Type'].map(mapeo_tipos).fillna('Other')

# --------------------------------
print(f"Limpieza completada. Quedan {df_clean.shape[0]} registros limpios.")

Limpieza completada. Quedan 142151 registros limpios.


## 4. Creación de variables objetivo (Targets)
Se definen las dos variables objetivo siguiendo las Reglas de Negocio del proyecto:
- **`Target_Retraso` (RN-01):** Retraso estructural crítico. Se marca como retraso si el tiempo real supera en un 40% la estimación **y** la tarea fue estimada en más de 90 minutos.
- **`Target_Riesgo` (RN-02):** Riesgo de alerta temprana. Se marca como riesgosa si su esfuerzo está en el Top 25% histórico **o** si es un Bug/Task con bloqueos activos.

In [11]:
# ==========================================================
# 3. CREACIÓN DE TARGETS (VARIABLES OBJETIVO)
# ==========================================================

# RN-01: Target_Retraso (Retraso Estructural Crítico)
# Se marca como retraso (1) SOLO si se cumplen dos condiciones estrictas:
# 1. El tiempo real supera a la estimación en más de un 40% (x1.40).
# 2. La tarea original se estimó en más de 1.5 horas (> 90 min).
# Propósito: Eliminar el ruido estadístico de las "micro-tareas" que se desvían 
# por minutos, enfocando al modelo en los retrasos que realmente rompen el Sprint.
df_clean['Target_Retraso'] = np.where(
    (df_clean['Resolution_Time_Minutes'] > (df_clean['Total_Effort_Minutes'] * 1.40)) & 
    (df_clean['Total_Effort_Minutes'] > 90), 1, 0
)


# RN-02: Target_Riesgo (Radar de Alerta Temprana)
# Se marca como tarea riesgosa (1) desde el Día 1 si cumple CUALQUIERA de estas condiciones:
# 1. Complejidad: Su esfuerzo total está en el Top 25% histórico de tareas más pesadas.
# 2. Bloqueo Estructural: Es un 'Bug' o 'Task' que ya nace con tickets bloqueadores vinculados.
# Propósito: Darle a la IA una señal fuerte para predecir problemas antes de que el tiempo empiece a correr.
umbral_esfuerzo = df_clean['Total_Effort_Minutes'].quantile(0.75)

df_clean['Target_Riesgo'] = np.where(
    (df_clean['Total_Effort_Minutes'] >= umbral_esfuerzo) | 
    ((df_clean['Blocker_Count'] > 0) & (df_clean['Issue_Type'].isin(['Bug', 'Task']))), 1, 0
)

# Mostrar resultados
print("--- Distribución de las Variables Objetivo ---")
print(f"Tasa real de Retrasos Críticos: {df_clean['Target_Retraso'].mean() * 100:.2f}%")
print(f"Tasa real de Riesgos: {df_clean['Target_Riesgo'].mean() * 100:.2f}%")

--- Distribución de las Variables Objetivo ---
Tasa real de Retrasos Críticos: 55.95%
Tasa real de Riesgos: 29.42%


## 5. Exportación del dataset limpio
Se exporta el dataset final a CSV para su consumo por los notebooks de EDA y modelado.

In [12]:
# 4. EXPORTACIÓN
output_path = '../data/processed/dataset_entrenamiento_tawos.csv'
df_clean.to_csv(output_path, index=False)
print(f"✅ Dataset limpio y exportado con éxito a: {output_path}")

✅ Dataset limpio y exportado con éxito a: ../data/processed/dataset_entrenamiento_tawos.csv
